# 01. Data Ingestion & Quality Validation Gates
### Energy Demand Forecasting Pipeline

This notebook handles the raw ingestion and validation of the Open Power System Data (OPSD) 60-minute time-series dataset.

**Key Operations:**
- Load raw OPSD singleindex CSV (~130 MB, 50,000+ hourly observations)
- Compute SHA-256 checksum for immutable data provenance and reproducibility
- Verify schema integrity, data types, and required target columns
- Check timestamp monotonicity, temporal continuity, and gap frequency
- Export validation report to `reports/validation_report.json`

## 1. Setup & Directory Paths

In [ ]:
import os
import sys
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

try:
    from IPython.display import display
except ImportError:
    display = print

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("energy_forecasting")

# Robust directory discovery
for base in [Path("."), Path(".."), Path("../..")]:
    candidate = base / "ml" / "data"
    if candidate.exists():
        PROJECT_ROOT = base.resolve()
        break
else:
    PROJECT_ROOT = Path(".").resolve()

ML_DIR = PROJECT_ROOT / "ml"
DATA_DIR = ML_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
ARTIFACTS_DIR = ML_DIR / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
FORECASTS_DIR = ARTIFACTS_DIR / "forecasts"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

for d in [PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, FORECASTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Locate raw 60min singleindex dataset
RAW_DATA_PATH = None
for candidate in [
    DATA_DIR / "opsd-time_series-2020-10-06" / "opsd-time_series-2020-10-06" / "time_series_60min_singleindex.csv",
    DATA_DIR / "time_series_60min_singleindex.csv",
    Path("ml/data/opsd-time_series-2020-10-06/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv"),
    Path("data/opsd-time_series-2020-10-06/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv"),
]:
    if candidate.exists():
        RAW_DATA_PATH = candidate.resolve()
        break

TIMESTAMP_COL = "utc_timestamp"
TARGET_COL_RAW = "DE_load_actual_entsoe_transparency"
TARGET_UNIT = "MW"
DS_COL = "ds"
Y_COL = "y"
TRAIN_RATIO = 0.70

print(f"Project root  : {PROJECT_ROOT}")
print(f"Raw data path : {RAW_DATA_PATH}")

Project root  : C:\Users\Soham\OneDrive\Desktop\Energy Demand Forecasting
Raw data path : C:\Users\Soham\OneDrive\Desktop\Energy Demand Forecasting\ml\data\opsd-time_series-2020-10-06\opsd-time_series-2020-10-06\time_series_60min_singleindex.csv


## 2. Ingestion & Cryptographic Checksum

In [ ]:
def sha256_checksum(filepath: Path) -> str:
    """Compute SHA-256 hex digest of a file in 64 KB chunks."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def load_opsd_timeseries(data_path: Path = RAW_DATA_PATH):
    """Load raw OPSD dataset and return dataframe with SHA256 checksum."""
    if not data_path or not data_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {data_path}")
    
    checksum = sha256_checksum(data_path)
    df = pd.read_csv(
        data_path,
        usecols=[TIMESTAMP_COL, TARGET_COL_RAW],
        dtype={TARGET_COL_RAW: "float64"},
        low_memory=False
    )
    return df, checksum

print("Loading raw OPSD dataset...")
df_raw, raw_checksum = load_opsd_timeseries(RAW_DATA_PATH)
print(f"Loaded {len(df_raw):,} records.")
print(f"SHA-256 Checksum: {raw_checksum}")
df_raw.head()

Loading raw OPSD dataset...
Loaded 50,401 records.
SHA-256 Checksum: 6a7f2bc571314cbf9c321cc03437691cd4be95c3a6f075e60ff99e8035c704c8


,utc_timestamp,DE_load_actual_entsoe_transparency
0,2014-12-31T23:00:00Z,NaN
1,2015-01-01T00:00:00Z,41151.0
2,2015-01-01T01:00:00Z,40135.0
3,2015-01-01T02:00:00Z,39106.0
4,2015-01-01T03:00:00Z,38765.0


## 3. Comprehensive Data Quality Validation

In [ ]:
def validate_dataset(df: pd.DataFrame):
    """Run comprehensive data validation checks."""
    results = {}
    
    # 1. Schema check
    has_cols = {col: col in df.columns for col in [TIMESTAMP_COL, TARGET_COL_RAW]}
    is_numeric = pd.api.types.is_numeric_dtype(df[TARGET_COL_RAW])
    results["schema"] = {
        "status": "PASS" if all(has_cols.values()) and is_numeric else "FAIL",
        "has_required_columns": has_cols,
        "target_is_numeric": bool(is_numeric)
    }
    
    # 2. Timestamps check
    ts = pd.to_datetime(df[TIMESTAMP_COL], utc=True, errors="coerce")
    is_monotonic = ts.is_monotonic_increasing
    null_ts = ts.isna().sum()
    results["timestamps"] = {
        "status": "PASS" if is_monotonic and null_ts == 0 else "FAIL",
        "is_monotonic_increasing": bool(is_monotonic),
        "invalid_timestamps": int(null_ts)
    }
    
    # 3. Missingness check
    target_nulls = int(df[TARGET_COL_RAW].isna().sum())
    target_null_pct = float(target_nulls / len(df) * 100)
    results["missingness"] = {
        "status": "PASS" if target_null_pct < 10.0 else "FAIL",
        "null_count": target_nulls,
        "null_pct": round(target_null_pct, 3)
    }
    
    # 4. Duplicates check
    dup_count = int(df[TIMESTAMP_COL].duplicated().sum())
    results["duplicates"] = {
        "status": "PASS" if dup_count == 0 else "WARNING",
        "duplicate_timestamps": dup_count
    }
    
    # 5. Gaps check
    diffs = ts.diff().dropna()
    expected_step = pd.Timedelta(hours=1)
    large_gaps = (diffs > expected_step).sum()
    results["gaps"] = {
        "status": "PASS" if large_gaps == 0 else "WARNING",
        "gaps_count": int(large_gaps)
    }
    
    return results

validation_report = validate_dataset(df_raw)
print(json.dumps(validation_report, indent=2))

report_file = REPORTS_DIR / "validation_report.json"
with open(report_file, "w") as f:
    json.dump(validation_report, f, indent=2)
print(f"Saved validation report to: {report_file}")

{
  "schema": {
    "status": "PASS",
    "has_required_columns": {
      "utc_timestamp": true,
      "DE_load_actual_entsoe_transparency": true
    },
    "target_is_numeric": true
  },
  "timestamps": {
    "status": "PASS",
    "is_monotonic_increasing": true,
    "invalid_timestamps": 0
  },
  "missingness": {
    "status": "PASS",
    "null_count": 1,
    "null_pct": 0.002
  },
  "duplicates": {
    "status": "PASS",
    "duplicate_timestamps": 0
  },
  "gaps": {
    "status": "PASS",
    "gaps_count": 0
  }
}
Saved validation report to: C:\Users\Soham\OneDrive\Desktop\Energy Demand Forecasting\reports\validation_report.json
